In [1]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
class SiameseSignatureDataset(Dataset):
    def __init__(self, root_dir, transform=None, mode='train'):
        """
        Args:
            root_dir (string): Directory with all the signature images.
            transform (callable, optional): Optional transform to be applied on an image.
            mode (string): Whether the dataset is in 'train' or 'test' mode.
        """
        self.root_dir = os.path.join(root_dir, mode)
        self.person_folders = [folder for folder in os.listdir(self.root_dir) if not folder.endswith('_forg')]
        self.transform = transform

    def __len__(self):
        return len(self.person_folders)

    def __getitem__(self, idx):
        """
        Returns a tuple (image1, image2, label) where:
        - image1, image2: Images of the two signatures.
        - label: 1 if similar (both real or both forged), 0 if dissimilar.
        """
        # Select a person's real and forged signature folders
        person_folder = self.person_folders[idx]
        real_folder = os.path.join(self.root_dir, person_folder)
        forg_folder = os.path.join(self.root_dir, person_folder + '_forg')
        
        real_signatures = os.listdir(real_folder)
        forged_signatures = os.listdir(forg_folder)

        # Decide randomly whether to create a positive (similar) or negative (dissimilar) pair
        is_positive_pair = random.choice([True, False])

        if is_positive_pair:
            # Positive pair: two real signatures or two forged signatures from the same person
            if random.choice([True, False]):
                # Pick two real signatures from the same person
                img1_name, img2_name = random.sample(real_signatures, 2)
                img1_path = os.path.join(real_folder, img1_name)
                img2_path = os.path.join(real_folder, img2_name)
            else:
                # Pick two forged signatures from the same person
                img1_name, img2_name = random.sample(forged_signatures, 2)
                img1_path = os.path.join(forg_folder, img1_name)
                img2_path = os.path.join(forg_folder, img2_name)

            label = 1  # Similar pair

        else:
            # Negative pair: one real and one forged signature from the same person
            img1_name = random.choice(real_signatures)
            img2_name = random.choice(forged_signatures)
            img1_path = os.path.join(real_folder, img1_name)
            img2_path = os.path.join(forg_folder, img2_name)

            label = 0  # Dissimilar pair

        # Load the images
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')

        # Apply transformations if provided
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

In [3]:
# Define transformations (resize, normalize, etc.)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Create dataset instances
train_dataset = SiameseSignatureDataset(root_dir='../raw_data/', transform=transform, mode='train')
test_dataset = SiameseSignatureDataset(root_dir='../raw_data/', transform=transform, mode='test')

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [4]:
from model_builder import SiameseSignatureCNN, SiameseNetwork, ContrastiveLoss

siamese_network = SiameseNetwork(SiameseSignatureCNN(3))
contrastive_loss = ContrastiveLoss()
optimizer = torch.optim.Adam(siamese_network.parameters(), lr=0.001)

In [5]:
num_epochs = 10
threshold = 0.5  # You can tune this based on your dataset

for epoch in range(num_epochs):
    siamese_network.train()  # Set the network to training mode
    total_loss = 0.0  # To accumulate total loss for this epoch
    correct_predictions = 0  # To count correct predictions for accuracy
    total_pairs = 0  # To keep track of the number of pairs

    for img1, img2, label in train_loader:
        optimizer.zero_grad()

        # Forward pass through the Siamese network
        distance = siamese_network(img1, img2)

        # Compute the contrastive loss
        loss = contrastive_loss(distance, label)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Accumulate loss for the epoch
        total_loss += loss.item()

        # Calculate accuracy
        predicted_labels = (distance < threshold).float()  # Predict label based on threshold
        correct_predictions += (predicted_labels == label).sum().item()  # Compare with actual labels
        total_pairs += label.size(0)  # Total number of pairs processed

    # Compute average loss and accuracy for this epoch
    avg_loss = total_loss / len(train_loader)  # Average loss
    accuracy = correct_predictions / total_pairs  # Accuracy for this epoch

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy * 100:.2f}%')

Epoch [1/10], Loss: 0.0000, Accuracy: 100.00%
Epoch [2/10], Loss: 0.1466, Accuracy: 0.00%
Epoch [3/10], Loss: 0.3292, Accuracy: 0.00%
Epoch [4/10], Loss: 14.3194, Accuracy: 0.00%
Epoch [5/10], Loss: 0.2939, Accuracy: 0.00%
Epoch [6/10], Loss: 0.2315, Accuracy: 0.00%
Epoch [7/10], Loss: 0.1661, Accuracy: 0.00%
Epoch [8/10], Loss: 0.4794, Accuracy: 0.00%
Epoch [9/10], Loss: 0.0945, Accuracy: 100.00%
Epoch [10/10], Loss: 0.2978, Accuracy: 0.00%


In [6]:
from sklearn.metrics import accuracy_score

siamese_network.eval()  # Set the model to evaluation mode
all_labels = []
all_predictions = []

with torch.no_grad():  # Disable gradient computation for evaluation
    for img1, img2, label in test_loader:
        # Forward pass
        distance = siamese_network(img1, img2)
        
        # Apply a threshold to get binary predictions (1 if similar, 0 if dissimilar)
        predictions = (distance < 0.5).float()  # Assuming 0.5 as the threshold
        
        all_labels.extend(label.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

# Calculate accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 100.00%


In [14]:
img3_path = '../raw_data/train/001/WhatsApp Image 2024-10-23 at 07.12.20_5abd3afc.jpg'
img3 = Image.open(img3_path).convert('RGB')
img3 = transform(img3)


img4_path = '../raw_data/train/001/WhatsApp Image 2024-10-23 at 07.12.20_88bcca97.jpg'
img4 = Image.open(img4_path).convert('RGB')
img4 = transform(img4)

In [15]:
img1, img2, label = next(iter(train_loader))
print(f"Image1 Shape: {img1.shape}, Image2 Shape: {img2.shape}, Label: {label}")

siamese_network.eval()
with torch.no_grad():
    distance = siamese_network(img4.unsqueeze(dim=0), img3.unsqueeze(dim=0))

threshold = 0.5
if distance.item() < threshold:
    print(f"The signatures are likely the same, with a distance of {distance.item()}")
else:
    print(f"The signatures are likely different, with a distance of {distance.item()}")

print(f"Distance between signatures: {distance.item()}")

Image1 Shape: torch.Size([1, 3, 224, 224]), Image2 Shape: torch.Size([1, 3, 224, 224]), Label: tensor([0.])
The signatures are likely the same, with a distance of 0.10558909177780151
Distance between signatures: 0.10558909177780151
